# Experiment: Xenium vs sc/snRNA-seq Sensitivity and 500-Gene Panel Design

## Objective
1. Align Xenium and sc/snRNA-seq feature spaces using shared genes.
2. Quantify and compare per-gene dynamic range and variability across assays.
3. Rank Xenium probes by empirical sensitivity.
4. Flag potentially problematic Xenium probes.
5. Propose a new 500-gene panel that distinguishes all subclasses (`adata.obs.subclass`) while minimizing redundancy.

## Deliverables
- `shared_gene_metrics.csv`
- `xenium_probe_sensitivity_ranking.csv`
- `xenium_problematic_probes.csv`
- `panel500_recommended_genes.csv`


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

from pathlib import Path
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(7)
warnings.filterwarnings('ignore')
sns.set_context('notebook')
sns.set_style('whitegrid')


## Plan

1. Load Xenium and sc/snRNA-seq `.h5ad` files.
2. Harmonize gene symbols and subset to shared genes for cross-assay comparison.
3. Compute assay-specific per-gene metrics (mean, detection rate, variance, robust dynamic range).
4. Build a Xenium sensitivity score and rank probes.
5. Flag probe failure modes (dropout vs scRNA, range compression, flat response, background-like pattern).
6. Use full-transcriptome sc/snRNA-seq to score subclass informativeness and robustness.
7. Select a 500-gene panel with subclass coverage and redundancy control.


In [ ]:
# Configuration
XENIUM_H5AD = Path('data/xenium.h5ad')
SCRNA_H5AD = Path('data/scrna_or_snrna.h5ad')
SUBCLASS_KEY = 'subclass'

ASSUME_INPUT_ALREADY_LOG1P = True
NORMALIZE_TARGET_SUM = 1e4

PANEL_SIZE = 500
SEED_GENES_PER_SUBCLASS = 4
REDUNDANCY_MAX_ABS_CORR = 0.90

MIN_DETECTION_RATE = 0.01
MIN_MEAN_EXPRESSION = 0.01
EXCLUDE_PREFIXES = ('MT-', 'RPL', 'RPS')

EPS = 1e-9
MAX_CELLS_FOR_QUANTILES = 50_000
RANDOM_SEED = 7

OUTDIR = Path('output/jupyter-notebook/results')
OUTDIR.mkdir(parents=True, exist_ok=True)

print('XENIUM_H5AD:', XENIUM_H5AD)
print('SCRNA_H5AD:', SCRNA_H5AD)
print('Output dir:', OUTDIR.resolve())


In [ ]:
# Helpers

def to_1d(x):
    return np.asarray(x).ravel()


def rank_pct(s: pd.Series) -> pd.Series:
    return s.rank(method='average', pct=True)


def robust_z(s: pd.Series) -> pd.Series:
    med = np.nanmedian(s.values)
    mad = np.nanmedian(np.abs(s.values - med))
    scale = 1.4826 * mad + EPS
    return (s - med) / scale


def standardize_gene_symbols(index: pd.Index) -> pd.Index:
    return pd.Index([str(g).strip().upper() for g in index])


def deduplicate_by_symbol(adata: ad.AnnData, symbol_col: str = 'gene_symbol') -> ad.AnnData:
    keep = ~adata.var[symbol_col].duplicated(keep='first')
    dropped = int((~keep).sum())
    if dropped > 0:
        print(f'Dropped {dropped} duplicated genes based on {symbol_col}.')
    out = adata[:, keep].copy()
    out.var_names = out.var[symbol_col].astype(str)
    out.var_names_make_unique()
    return out


def maybe_log_normalize(adata: ad.AnnData, assume_log1p: bool = True) -> ad.AnnData:
    out = adata.copy()
    if assume_log1p:
        return out
    sc.pp.normalize_total(out, target_sum=NORMALIZE_TARGET_SUM)
    sc.pp.log1p(out)
    return out


def gene_quantile_range(adata: ad.AnnData, q_low: float = 0.05, q_high: float = 0.95, max_cells: int = 50_000, seed: int = 7):
    n = adata.n_obs
    rng = np.random.default_rng(seed)
    if n > max_cells:
        idx = np.sort(rng.choice(n, size=max_cells, replace=False))
        X = adata.X[idx, :]
    else:
        X = adata.X

    if sparse.issparse(X):
        X = X.toarray()

    ql = np.quantile(X, q_low, axis=0)
    qh = np.quantile(X, q_high, axis=0)
    return qh - ql, ql, qh


def summarize_genes(adata: ad.AnnData, assay_name: str, include_quantiles: bool = True) -> pd.DataFrame:
    X = adata.X
    n_cells = adata.n_obs

    if sparse.issparse(X):
        mean_expr = to_1d(X.mean(axis=0))
        sq_mean = to_1d(X.power(2).mean(axis=0))
        det_rate = X.getnnz(axis=0) / n_cells
    else:
        mean_expr = np.mean(X, axis=0)
        sq_mean = np.mean(np.square(X), axis=0)
        det_rate = np.mean(X > 0, axis=0)

    var_expr = np.maximum(sq_mean - np.square(mean_expr), 0)

    out = pd.DataFrame({
        'gene': adata.var_names.astype(str),
        f'mean_{assay_name}': mean_expr,
        f'var_{assay_name}': var_expr,
        f'detection_rate_{assay_name}': det_rate,
    })

    if include_quantiles:
        dr, q05, q95 = gene_quantile_range(
            adata,
            q_low=0.05,
            q_high=0.95,
            max_cells=MAX_CELLS_FOR_QUANTILES,
            seed=RANDOM_SEED,
        )
        out[f'dynamic_range_{assay_name}'] = dr
        out[f'q05_{assay_name}'] = q05
        out[f'q95_{assay_name}'] = q95

    return out


def subclass_means_matrix(adata: ad.AnnData, group_key: str) -> pd.DataFrame:
    groups = adata.obs[group_key].astype('category')
    labels = groups.cat.categories.tolist()
    codes = groups.cat.codes.to_numpy()

    n_groups = len(labels)
    n_cells = adata.n_obs

    one_hot = sparse.csr_matrix(
        (np.ones(n_cells, dtype=np.float32), (codes, np.arange(n_cells))),
        shape=(n_groups, n_cells),
    )

    sums = one_hot @ adata.X
    counts = to_1d(one_hot.sum(axis=1))
    counts = np.maximum(counts, 1)

    if sparse.issparse(sums):
        means = sums.multiply(1.0 / counts[:, None]).toarray()
    else:
        means = sums / counts[:, None]

    return pd.DataFrame(means, index=labels, columns=adata.var_names.astype(str))


def max_abs_corr_to_selected(gene_idx: int, selected_idx: list[int], zmat: np.ndarray) -> float:
    if not selected_idx:
        return 0.0
    v = zmat[:, gene_idx]
    S = zmat[:, selected_idx]
    corr = np.abs((v[:, None] * S).mean(axis=0))
    return float(np.max(corr))


In [ ]:
# Load data and harmonize gene symbols
if not XENIUM_H5AD.exists() or not SCRNA_H5AD.exists():
    raise FileNotFoundError(
        'Set XENIUM_H5AD and SCRNA_H5AD to valid files before running this notebook.'
    )

xenium = ad.read_h5ad(XENIUM_H5AD)
scrna = ad.read_h5ad(SCRNA_H5AD)

if SUBCLASS_KEY not in scrna.obs.columns:
    raise KeyError(f'{SUBCLASS_KEY!r} not found in sc/snRNA obs columns.')

xenium.var['gene_symbol'] = standardize_gene_symbols(xenium.var_names)
scrna.var['gene_symbol'] = standardize_gene_symbols(scrna.var_names)

xenium = deduplicate_by_symbol(xenium, 'gene_symbol')
scrna = deduplicate_by_symbol(scrna, 'gene_symbol')

xenium = maybe_log_normalize(xenium, assume_log1p=ASSUME_INPUT_ALREADY_LOG1P)
scrna = maybe_log_normalize(scrna, assume_log1p=ASSUME_INPUT_ALREADY_LOG1P)

shared_genes = sorted(set(xenium.var_names) & set(scrna.var_names))
print('Xenium cells x genes:', xenium.shape)
print('sc/snRNA cells x genes:', scrna.shape)
print('Shared genes:', len(shared_genes))


In [ ]:
# Shared-gene metrics: dynamic range, variability, and assay concordance
if len(shared_genes) == 0:
    raise ValueError('No shared genes found after harmonization.')

xenium_shared = xenium[:, shared_genes].copy()
scrna_shared = scrna[:, shared_genes].copy()

x_metrics = summarize_genes(xenium_shared, assay_name='xenium', include_quantiles=True)
r_metrics = summarize_genes(scrna_shared, assay_name='scrna', include_quantiles=True)

shared_metrics = x_metrics.merge(r_metrics, on='gene', how='inner')

shared_metrics['dynamic_range_ratio_xenium_over_scrna'] = (
    (shared_metrics['dynamic_range_xenium'] + EPS) /
    (shared_metrics['dynamic_range_scrna'] + EPS)
)
shared_metrics['variance_ratio_xenium_over_scrna'] = (
    (shared_metrics['var_xenium'] + EPS) /
    (shared_metrics['var_scrna'] + EPS)
)
shared_metrics['detection_log2_ratio_xenium_over_scrna'] = np.log2(
    (shared_metrics['detection_rate_xenium'] + EPS) /
    (shared_metrics['detection_rate_scrna'] + EPS)
)

shared_metrics.head()


In [ ]:
# Visualize shared-gene assay behavior
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(
    data=shared_metrics,
    x='detection_rate_scrna',
    y='detection_rate_xenium',
    alpha=0.7,
    s=40,
    ax=axes[0],
)
axes[0].set_title('Detection rate: sc/snRNA vs Xenium')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

sns.scatterplot(
    data=shared_metrics,
    x='dynamic_range_scrna',
    y='dynamic_range_xenium',
    alpha=0.7,
    s=40,
    ax=axes[1],
)
axes[1].set_title('Dynamic range (q95 - q05)')

sns.scatterplot(
    data=shared_metrics,
    x='var_scrna',
    y='var_xenium',
    alpha=0.7,
    s=40,
    ax=axes[2],
)
axes[2].set_title('Variance')

plt.tight_layout()

shared_metrics[['dynamic_range_ratio_xenium_over_scrna', 'variance_ratio_xenium_over_scrna']].describe()


In [ ]:
# Xenium probe sensitivity scoring and ranking
probe_scores = shared_metrics.copy()

# Concordance favors probes where Xenium and sc/snRNA detection rates are similar.
probe_scores['detection_concordance'] = -np.abs(
    np.log2((probe_scores['detection_rate_xenium'] + EPS) / (probe_scores['detection_rate_scrna'] + EPS))
)

probe_scores['sensitivity_score'] = (
    0.35 * robust_z(probe_scores['detection_rate_xenium']) +
    0.25 * robust_z(probe_scores['dynamic_range_xenium']) +
    0.20 * robust_z(probe_scores['var_xenium']) +
    0.20 * robust_z(probe_scores['detection_concordance'])
)

probe_scores = probe_scores.sort_values('sensitivity_score', ascending=False).reset_index(drop=True)
probe_scores['sensitivity_rank'] = np.arange(1, len(probe_scores) + 1)

probe_scores[['gene', 'sensitivity_rank', 'sensitivity_score']].head(20)


In [ ]:
# Flag potentially problematic Xenium probes
ps = probe_scores.copy()

ps['flag_dropout_vs_scrna'] = (
    (ps['detection_rate_scrna'] >= 0.20) &
    (ps['detection_rate_xenium'] <= 0.05)
)
ps['flag_range_compression'] = (
    (ps['dynamic_range_scrna'] >= ps['dynamic_range_scrna'].median()) &
    (ps['dynamic_range_ratio_xenium_over_scrna'] <= 0.35)
)
ps['flag_flat_xenium_response'] = (
    (ps['detection_rate_xenium'] >= 0.10) &
    (ps['var_xenium'] <= ps['var_xenium'].quantile(0.10))
)
ps['flag_background_like'] = (
    (ps['detection_rate_xenium'] >= 0.95) &
    (ps['detection_rate_scrna'] <= 0.05)
)

flag_cols = [
    'flag_dropout_vs_scrna',
    'flag_range_compression',
    'flag_flat_xenium_response',
    'flag_background_like',
]

ps['n_flags'] = ps[flag_cols].sum(axis=1)
ps['problem_flags'] = ps[flag_cols].apply(
    lambda row: ', '.join([c for c, v in row.items() if v]) if row.any() else '',
    axis=1,
)

problematic_probes = ps[ps['n_flags'] > 0].sort_values(['n_flags', 'sensitivity_score'], ascending=[False, True])
problematic_probes[['gene', 'n_flags', 'problem_flags', 'sensitivity_score']].head(30)


## 500-gene panel design using full-transcriptome sc/snRNA-seq

Design principles:
1. **Informative for subclass separation**: prioritize genes with high one-vs-rest pseudobulk separation.
2. **Robust expression**: avoid genes with low mean expression or very low detection.
3. **Empirical Xenium prior where available**: genes already sensitive in Xenium get a boost.
4. **Redundancy control**: greedy selection with max absolute pseudobulk-correlation threshold.
5. **Coverage guarantee**: seed each subclass with top non-redundant markers before global fill.


In [ ]:
# Full-transcriptome sc/snRNA gene robustness and subclass pseudobulk
sc_full_stats = summarize_genes(scrna, assay_name='scrna_full', include_quantiles=False).set_index('gene')

pb_means = subclass_means_matrix(scrna, SUBCLASS_KEY)

if pb_means.shape[0] < 2:
    raise ValueError('Need at least two subclasses to design a discriminatory panel.')

# Basic filtering for non-informative genes.
candidate_mask = (
    (sc_full_stats['detection_rate_scrna_full'] >= MIN_DETECTION_RATE) &
    (sc_full_stats['mean_scrna_full'] >= MIN_MEAN_EXPRESSION)
)

if EXCLUDE_PREFIXES:
    excluded = pd.Series(sc_full_stats.index).str.startswith(EXCLUDE_PREFIXES).values
    candidate_mask = candidate_mask & (~excluded)

candidate_genes = sc_full_stats.index[candidate_mask].tolist()
print('Candidate genes after basic filtering:', len(candidate_genes))


In [ ]:
# Compute subclass-separation scores and aggregate panel score
pb = pb_means[candidate_genes]
pb_values = pb.values
classes = pb.index.tolist()
genes = pb.columns.tolist()

n_classes, n_genes = pb_values.shape
class_sum = pb_values.sum(axis=0)

class_scores = np.zeros((n_classes, n_genes), dtype=np.float64)
for i in range(n_classes):
    in_mean = pb_values[i, :]
    out_mean = (class_sum - in_mean) / max(n_classes - 1, 1)
    class_scores[i, :] = np.log2((in_mean + EPS) / (out_mean + EPS)) * np.log1p(in_mean)

best_class_idx = np.argmax(class_scores, axis=0)
best_marker_score = class_scores.max(axis=0)

# Robustness and spread terms from sc/snRNA.
mean_expr = sc_full_stats.loc[genes, 'mean_scrna_full']
det_rate = sc_full_stats.loc[genes, 'detection_rate_scrna_full']
robust_expr_score = rank_pct(mean_expr) * 0.5 + rank_pct(det_rate) * 0.5

spread_score = rank_pct(pd.Series(pb_values.max(axis=0) - pb_values.min(axis=0), index=genes))
marker_score = rank_pct(pd.Series(best_marker_score, index=genes))

# Xenium empirical prior when gene is in the current Xenium panel; neutral otherwise.
xen_prior = pd.Series(0.5, index=genes)
if 'sensitivity_score' in probe_scores.columns:
    shared_prior = rank_pct(probe_scores.set_index('gene')['sensitivity_score'])
    overlap = xen_prior.index.intersection(shared_prior.index)
    xen_prior.loc[overlap] = shared_prior.loc[overlap]

panel_score = (
    0.45 * marker_score +
    0.25 * robust_expr_score +
    0.20 * xen_prior +
    0.10 * spread_score
)

panel_table = pd.DataFrame({
    'gene': genes,
    'panel_score': panel_score.values,
    'best_subclass': [classes[i] for i in best_class_idx],
    'best_marker_score': best_marker_score,
    'mean_scrna_full': mean_expr.values,
    'detection_rate_scrna_full': det_rate.values,
    'xenium_overlap': pd.Index(genes).isin(set(shared_genes)).astype(int),
    'xenium_prior': xen_prior.values,
}).sort_values('panel_score', ascending=False)

panel_table.head(20)


In [ ]:
# Redundancy-aware greedy selection with subclass seeding
class_score_df = pd.DataFrame(class_scores, index=classes, columns=genes)

# z-score subclass profiles per gene for fast correlation checks.
pb_z = pb.copy()
pb_z = (pb_z - pb_z.mean(axis=0)) / (pb_z.std(axis=0) + EPS)
pb_z = pb_z.fillna(0.0)
zmat = pb_z.values  # shape: subclasses x genes

idx_of = {g: i for i, g in enumerate(genes)}
selected: list[str] = []
selected_idx: list[int] = []

# Step 1: ensure subclass coverage.
for cls in classes:
    ranked = class_score_df.loc[cls].sort_values(ascending=False).index.tolist()
    picked = 0
    for g in ranked:
        if g in selected:
            continue
        gi = idx_of[g]
        if max_abs_corr_to_selected(gi, selected_idx, zmat) <= REDUNDANCY_MAX_ABS_CORR:
            selected.append(g)
            selected_idx.append(gi)
            picked += 1
        if picked >= SEED_GENES_PER_SUBCLASS:
            break

# Step 2: global fill by panel score.
global_ranked = panel_table['gene'].tolist()
for g in global_ranked:
    if len(selected) >= PANEL_SIZE:
        break
    if g in selected:
        continue
    gi = idx_of[g]
    if max_abs_corr_to_selected(gi, selected_idx, zmat) <= REDUNDANCY_MAX_ABS_CORR:
        selected.append(g)
        selected_idx.append(gi)

# Step 3: relaxed fill if still short.
relaxed = REDUNDANCY_MAX_ABS_CORR
while len(selected) < PANEL_SIZE and relaxed < 0.99:
    relaxed += 0.02
    for g in global_ranked:
        if len(selected) >= PANEL_SIZE:
            break
        if g in selected:
            continue
        gi = idx_of[g]
        if max_abs_corr_to_selected(gi, selected_idx, zmat) <= relaxed:
            selected.append(g)
            selected_idx.append(gi)

panel500 = panel_table.set_index('gene').loc[selected].reset_index().head(PANEL_SIZE).copy()
panel500['panel_rank'] = np.arange(1, len(panel500) + 1)

print('Selected genes:', len(panel500))
panel500.head(20)


In [ ]:
# Diagnostics: subclass coverage and redundancy summary
coverage = panel500['best_subclass'].value_counts().rename('n_panel_genes').to_frame()
coverage = coverage.reindex(pb_means.index, fill_value=0)

print('Subclass coverage (first 20):')
display(coverage.head(20))

if len(panel500) > 1:
    sel_pb = pb_means[panel500['gene']]
    sel_corr = sel_pb.corr().abs()
    upper = sel_corr.where(np.triu(np.ones(sel_corr.shape), k=1).astype(bool))
    max_pairwise_corr = np.nanmax(upper.values)
    print('Max pairwise absolute correlation among selected genes:', round(float(max_pairwise_corr), 4))

# Optional quick heatmap of top 60 genes by panel rank.
heat_n = min(60, len(panel500))
if heat_n > 0:
    hm = pb_means[panel500.sort_values('panel_rank')['gene'].head(heat_n)]
    hm_z = (hm - hm.mean(axis=0)) / (hm.std(axis=0) + EPS)
    plt.figure(figsize=(14, max(4, 0.22 * hm_z.shape[0])))
    sns.heatmap(hm_z, cmap='vlag', center=0)
    plt.title(f'Subclass pseudobulk z-score (top {heat_n} panel genes)')
    plt.xlabel('Genes')
    plt.ylabel(SUBCLASS_KEY)
    plt.tight_layout()


In [ ]:
# Save outputs
shared_metrics_out = OUTDIR / 'shared_gene_metrics.csv'
probe_rank_out = OUTDIR / 'xenium_probe_sensitivity_ranking.csv'
problematic_out = OUTDIR / 'xenium_problematic_probes.csv'
panel_out = OUTDIR / 'panel500_recommended_genes.csv'

shared_metrics.sort_values('gene').to_csv(shared_metrics_out, index=False)
probe_scores.to_csv(probe_rank_out, index=False)
problematic_probes.to_csv(problematic_out, index=False)
panel500.to_csv(panel_out, index=False)

summary = {
    'n_xenium_genes': int(xenium.n_vars),
    'n_scrna_genes': int(scrna.n_vars),
    'n_shared_genes': int(len(shared_genes)),
    'n_problematic_probes': int(problematic_probes.shape[0]),
    'n_panel_genes_selected': int(panel500.shape[0]),
    'outputs': {
        'shared_metrics': str(shared_metrics_out),
        'probe_ranking': str(probe_rank_out),
        'problematic_probes': str(problematic_out),
        'panel500': str(panel_out),
    },
}
summary


## Next steps

1. Inspect top and bottom ranked Xenium probes from `xenium_probe_sensitivity_ranking.csv`.
2. Manually review `xenium_problematic_probes.csv` against known assay QC notes.
3. Validate `panel500_recommended_genes.csv` with held-out cells/donors and subclass classification performance.
4. If needed, tune weights (`panel_score`) and redundancy threshold (`REDUNDANCY_MAX_ABS_CORR`) and re-run.

### Validation note
If this notebook is not executed in this environment, run all cells top-to-bottom locally after updating the two input file paths.
